In [3]:
# rlhf.py
# ─────────────────────────────────────────
# RLHF — Stage 3
# Loads the SFT model as the starting point.
# This is the most important detail:
#   Pre-trained model → SFT model → RLHF model
# You never run RLHF on a raw pre-trained model.
# The SFT model already knows how to follow
# instructions — RLHF just refines the behaviour.
# ─────────────────────────────────────────
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from model import BigramLanguageModel


# ─────────────────────────────────────────
# 1. Load tokenizer
# Same tokenizer used in pre-training and SFT.
# All three stages MUST share the same vocabulary —
# the token ids must mean the same thing throughout.
# ─────────────────────────────────────────
with open("pre-trained-model/tokenizer.json", "r", encoding="utf-8") as f:
    tokenizer = json.load(f)

stoi = tokenizer["stoi"]
itos = {int(k): v for k, v in tokenizer["itos"].items()}

encode = lambda s: [stoi[c] for c in s if c in stoi]   # safe encode — skips unknowns
decode = lambda l: ''.join([itos[i] for i in l])


# ─────────────────────────────────────────
# 2. Load config
# Architecture must be identical to SFT.
# The RLHF model is not a new model —
# it is the SFT model with adjusted weights.
# ─────────────────────────────────────────
with open("pre-trained-model/config.json", "r") as f:
    config = json.load(f)

vocab_size = config["vocab_size"]
print(f"Model type:      {config['model_type']}")
print(f"Vocab size:      {vocab_size}")


# ─────────────────────────────────────────
# 3. Load the SFT model — NOT the pre-trained one
# This is the critical distinction:
#
#   pretrain.py  →  bigram_pretrained.pt   (base model)
#   finetune.py  →  sft-model/bigram_sft.pt  (instruction model)  ← load this
#   rlhf.py      →  rlhf-model/bigram_rlhf.pt (aligned model)     ← produces this
#
# In HuggingFace terms:
#   AutoModel.from_pretrained("sft-model/")
# ─────────────────────────────────────────
m = BigramLanguageModel(vocab_size)
m.load_state_dict(torch.load("sft-model/bigram_sft.pt"))
m.train()   # set to train mode — RLHF will update its weights
print("SFT model loaded successfully — starting RLHF from SFT checkpoint")


# ─────────────────────────────────────────
# 4. Freeze a reference copy of the SFT model
# In real PPO this is called the "reference policy".
# It is used to compute the KL penalty —
# a measure of how far the RL model has drifted
# from the SFT model. Without it, the model
# would reward-hack: score high on the reward
# model while producing nonsense text.
#
# We freeze it so its weights never update.
# ─────────────────────────────────────────
import copy

m_ref = copy.deepcopy(m)        # identical copy of the SFT model
for param in m_ref.parameters():
    param.requires_grad = False  # frozen — reference only, never trained
m_ref.eval()
print("Reference policy frozen")


# ─────────────────────────────────────────
# Ready — proceed to Stage 3a, 3b, 3c below
# ─────────────────────────────────────────
print(f"\nAll assets loaded:")
print(f"  Tokenizer : trained-model/tokenizer.json  ({vocab_size} tokens)")
print(f"  Config    : trained-model/config.json")
print(f"  SFT model : sft-model/bigram_sft.pt       (active, will be updated)")
print(f"  Ref model : copy of SFT                   (frozen, KL anchor)")

Model type:      BigramLanguageModel
Vocab size:      65
SFT model loaded successfully — starting RLHF from SFT checkpoint
Reference policy frozen

All assets loaded:
  Tokenizer : trained-model/tokenizer.json  (65 tokens)
  Config    : trained-model/config.json
  SFT model : sft-model/bigram_sft.pt       (active, will be updated)
  Ref model : copy of SFT                   (frozen, KL anchor)


In [7]:
# ─────────────────────────────────────────
# RLHF Stage 3a: Generate candidate responses
# For each prompt, the model produces N responses.
# Humans (or a heuristic) then rank them.
# In real RLHF: crowdworkers rank on helpfulness,
# harmlessness, honesty (the 3H criteria)
# ─────────────────────────────────────────

prompts = [
    "### Instruction:\nWhat is gravity?\n\n### Response:\n",
    "### Instruction:\nWhat is the capital of France?\n\n### Response:\n",
]

def generate_response(model, prompt, max_new_tokens=80, temperature=1.0):
    context = torch.tensor([encode(prompt)], dtype=torch.long)
    output  = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature)
    # Return only the newly generated part (after the prompt)
    return decode(output[0][len(encode(prompt)):].tolist())

# Generate 3 candidate responses per prompt at different temperatures
# Temperature controls randomness: low = conservative, high = creative
candidates = {}
for prompt in prompts:
    candidates[prompt] = [
        generate_response(m, prompt, temperature=0.5),   # conservative
        generate_response(m, prompt, temperature=1.0),   # balanced
        generate_response(m, prompt, temperature=1.5),   # creative/wild
    ]
    print(f"\nPrompt: {prompt}...")
    for i, r in enumerate(candidates[prompt]):
        print(f"  Candidate {i}: {repr(r)}")


Prompt: ### Instruction:
What is gravity?

### Response:
...
  Candidate 0: "IEMhofAZrnFY,sBeweBuo oARD3kJ'SPq, tZNTInt hylaX'ind lafkewaWecOseUHhoZ!kJ.KLWP3"
  Candidate 1: "K'CWLyet,MPm koZMhT:igN:3CHGkoftnvK?Es!'w,M\ncgrn oRFYbMPp?\nT-lA,\n\n-ss-A,V'S,V:AQ"
  Candidate 2: 'UTUY$!koVlFHgMHl wUNCofJGqE mfesXDsD$-sjbjIAo,pxLG;AIGwU-a EruLWodk b!Q,oSn.IVGs'

Prompt: ### Instruction:
What is the capital of France?

### Response:
...
  Candidate 0: 'ORyof woOUMyqYoRAEOd thasecV:3;\ncoRicuVOUNqg.dyWxfkJ$UMq, bt JhUMy&BithoRIAst cO'
  Candidate 1: 'TulasoaY$unle.P,SrokyLs,urrtsBrYQDIpMhTtN:!V33PuzTOdy&Hvqg&,r-bj?V3E:3nUFYL??chy'
  Candidate 2: "OGCgxsiUDGpqJp3Uf?.aa3dddHX&CUzKMPoMgdnsX?!ivrMgTOfEIDT$3UE\nTsutjOX;OBHg:y,Sm:n'"


In [5]:
# ─────────────────────────────────────────
# RLHF Stage 3b: Reward Model
# A separate model that scores a response.
# High score = humans would prefer this.
# Low score  = humans would reject this.
#
# In real RLHF (InstructGPT): the reward model
# is a copy of the SFT model with the final
# layer replaced by a scalar output head.
#
# Here we simulate it with:
#   - A simple scoring heuristic (stands in for human labels)
#   - A small neural network that learns to predict the score
# ─────────────────────────────────────────

class RewardModel(nn.Module):
    """
    Takes a sequence of tokens, embeds them,
    averages the embeddings, and outputs a
    single scalar score.

    Real reward models are full transformers
    with a linear head on the [EOS] token.
    """
    def __init__(self, vocab_size, embed_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.scorer    = nn.Sequential(
            nn.Linear(embed_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)          # single scalar output
        )

    def forward(self, idx):
        # idx shape: (B, T)
        embedded = self.embedding(idx)       # (B, T, embed_dim)
        pooled   = embedded.mean(dim=1)      # (B, embed_dim) — average pooling
        score    = self.scorer(pooled)       # (B, 1)
        return score.squeeze(-1)             # (B,)


# ── Simulate human preference labels ──────
# In real RLHF humans rank responses A > B > C.
# We simulate this with a heuristic:
#   longer responses with more unique chars = better quality
def human_preference_score(response: str) -> float:
    """
    Simulated human score (0.0 to 1.0).
    Real version: crowdworker rankings.
    """
    length_score    = min(len(response) / 100, 1.0)    # longer is better (up to 100 chars)
    diversity_score = len(set(response)) / vocab_size  # more unique chars = richer response
    return (length_score + diversity_score) / 2.0


# ── Build reward training dataset ─────────
reward_inputs  = []
reward_targets = []

for prompt, responses in candidates.items():
    for response in responses:
        full_text = prompt + response
        tokens    = encode(full_text)[:128]        # cap at 128 tokens
        tokens   += [0] * (128 - len(tokens))      # pad to fixed length
        reward_inputs.append(tokens)
        reward_targets.append(human_preference_score(response))

reward_inputs  = torch.tensor(reward_inputs,  dtype=torch.long)
reward_targets = torch.tensor(reward_targets, dtype=torch.float)
print(f"Reward training samples: {len(reward_inputs)}")
print(f"Score range: {reward_targets.min():.2f} → {reward_targets.max():.2f}")


# ── Train the Reward Model ─────────────────
rm            = RewardModel(vocab_size)
rm_optimizer  = torch.optim.Adam(rm.parameters(), lr=1e-3)
rm_loss_fn    = nn.MSELoss()        # predict a scalar score → regression

print("\nTraining Reward Model...")
for step in range(300):
    predicted = rm(reward_inputs)
    loss      = rm_loss_fn(predicted, reward_targets)

    rm_optimizer.zero_grad()
    loss.backward()
    rm_optimizer.step()

    if step % 100 == 0:
        print(f"  Step {step:3d} | RM loss: {loss.item():.4f}")

print("Reward Model trained.")

Reward training samples: 6
Score range: 0.71 → 0.76

Training Reward Model...
  Step   0 | RM loss: 0.5856
  Step 100 | RM loss: 0.0018
  Step 200 | RM loss: 0.0005
Reward Model trained.


In [6]:
# ─────────────────────────────────────────
# RLHF Stage 3c: RL Fine-tuning (simplified PPO)
# Use the reward model's score as a training signal
# to update the language model's weights.
#
# Real PPO is complex (clipped surrogate objective,
# value network, KL penalty vs reference model).
# Here we simulate the core idea:
#   1. Sample a response from the LM
#   2. Score it with the reward model
#   3. Use (1 - reward) as a loss signal
#   4. Backprop into the LM
#
# The KL penalty (not shown) would prevent the
# LM from drifting too far from the SFT model.
# ─────────────────────────────────────────

rl_optimizer = torch.optim.AdamW(m.parameters(), lr=1e-5)   # very low LR for RL stage

print("\nRL Fine-tuning with reward signal...")
print(f"{'Step':>6}  {'Reward':>8}  {'RL Loss':>10}")
print("-" * 32)

for step in range(200):

    # 1. Pick a random prompt
    prompt      = prompts[step % len(prompts)]
    prompt_ids  = encode(prompt)
    context     = torch.tensor([prompt_ids], dtype=torch.long)

    # 2. Sample a response from the current LM (with gradient)
    response_ids = []
    for _ in range(40):                                # generate 40 tokens
        logits, _ = m(context)
        logits    = logits[:, -1, :]                  # last time step
        probs     = torch.softmax(logits, dim=-1)
        next_id   = torch.multinomial(probs, 1)
        context   = torch.cat([context, next_id], dim=1)
        response_ids.append(next_id.item())

    # 3. Score the full sequence with the Reward Model
    full_ids = prompt_ids + response_ids
    full_ids = full_ids[:128] + [0] * max(0, 128 - len(full_ids))   # pad
    rm_input = torch.tensor([full_ids], dtype=torch.long)

    with torch.no_grad():
        reward = rm(rm_input).item()        # scalar score — no grad through RM

    # 4. RL loss = -reward (maximise reward = minimise negative reward)
    #    We re-run the LM to get a differentiable loss
    logits, lm_loss = m(
        torch.tensor([prompt_ids], dtype=torch.long),
        torch.tensor([prompt_ids[1:] + [0]], dtype=torch.long)
    )
    rl_loss = (1.0 - reward) * lm_loss     # scale LM loss by inverse reward
                                            # high reward → small update
                                            # low reward  → large update

    rl_optimizer.zero_grad()
    rl_loss.backward()
    rl_optimizer.step()

    if step % 50 == 0:
        print(f"{step:>6}  {reward:>8.4f}  {rl_loss.item():>10.4f}")


# ─────────────────────────────────────────
# Save the RLHF-aligned model
# This is your "ChatGPT" checkpoint —
# the model that is helpful, not just fluent
# ─────────────────────────────────────────
import os
os.makedirs("rlhf-model", exist_ok=True)
torch.save(m.state_dict(), "rlhf-model/bigram_rlhf.pt")
torch.save(rm.state_dict(), "rlhf-model/reward_model.pt")
print("\nRLHF model saved to rlhf-model/")


RL Fine-tuning with reward signal...
  Step    Reward     RL Loss
--------------------------------
     0    1.0185     -0.0666
    50    1.0370     -0.1328
   100    0.9888      0.0401
   150    1.0291     -0.1047

RLHF model saved to rlhf-model/
